# Boruta

## Demo with a histomorphology dataset

The [UCI ML Breast Cancer Wisconsin (Diagnostic) dataset](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) present features computed from a digitized image of a fine needle aspirate (FNA) of a breast mass. They describe characteristics of the cell nuclei present in the image.

The goal is to determine if the mass is malignant (0) or benign (1).

In [ ]:
from sklearn.datasets import load_breast_cancer
X,y = load_breast_cancer(return_X_y = True, as_frame = True)
X.head()

In [ ]:
from sklearn.feature_selection import chi2
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math


## compute chi2 statistic
chi2_stat , chi2_pv = chi2( X,y )

## order features by chi2
sorted_features = X.columns[  np.argsort(chi2_pv) ]

## plotting
nrows = math.ceil(len(sorted_features)/3)
fig,axes  = plt.subplots( nrows,3 , figsize = (15,nrows*5) )
for i,feature in enumerate(sorted_features):
    
    ax = axes[ i//3 , i%3 ]
    
    sns.boxplot( x = y , y = X[feature] , ax = ax )
    ax.set_title(feature)
    
fig.tight_layout()

## training a Random Forest

For the sake of time, we will only optimize the ccp_alpha pruning parameter

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline


rf = RandomForestClassifier(n_jobs=-1)

rf_grid = GridSearchCV(rf,
                       {"ccp_alpha" : np.logspace(-5,-2,20)},
                       scoring='accuracy'
                      )
%time rf_grid.fit(X,y)

In [ ]:
rf_grid.best_params_

In [ ]:
rf_grid.best_score_

In [ ]:
RF_fi = rf_grid.best_estimator_.feature_importances_
print( f"{( RF_fi != 0  ).sum()}/{X.shape[1]} features with non-zero importances")

## training a Linear Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


ppl = Pipeline([('scale' , StandardScaler()),
                ('model' , LogisticRegression(l1_ratio=1.0, solver = 'liblinear'))
               ])

lr_grid = GridSearchCV(ppl,
                       {"model__C" : np.logspace(-5,5,100)},
                       scoring='accuracy'
                      )
%time lr_grid.fit(X,y)

In [ ]:
lr_grid.best_params_

In [ ]:
lr_grid.best_score_

In [ ]:
LR_fi = lr_grid.best_estimator_['model'].coef_[0]
num_non_null = ( LR_fi != 0 ).sum()

print( f"{num_non_null}/{X.shape[1]} features with non-zero importances")

## Borutapy

In [ ]:
from boruta import BorutaPy

rf = RandomForestClassifier(n_jobs=-1, ccp_alpha = rf_grid.best_params_["ccp_alpha"])

feat_selector = BorutaPy(rf, n_estimators='auto', verbose=2, random_state=123, max_iter = 250 )

In [ ]:
%time feat_selector.fit(X, y)

In [ ]:
feat_selector.support_

In [ ]:
feat_selector.support_weak_

In [ ]:
feat_selector.estimator

In [ ]:
from sklearn.model_selection import cross_val_score
## keep only the confirmed
Xf = feat_selector.transform( np.array(X) )
%time cross_val_score(rf, Xf,y, scoring='accuracy').mean()

In [ ]:
## keep only confirmed + tentative
Xf = feat_selector.transform( np.array(X) , weak = True )
%time cross_val_score(rf, Xf,y, scoring='accuracy').mean()

In [ ]:
import pandas as pd
df_fs = pd.DataFrame( {"LR":LR_fi!=0 , "Boruta": feat_selector.support_ } , index = X.columns  )
df_fs.sort_values(by="LR", ascending=False)


In [ ]:
## just for fun, let's train a model on the features selected by Boruta and not by the LR
features_Boruta_not_LR = df_fs.index[ ( ~df_fs.LR  ) & df_fs.Boruta ]

%time cross_val_score(rf, X[features_Boruta_not_LR],y, scoring='accuracy').mean()

We can see that the features that were not selected by the L1 logistic regression, but kept by Boruta actually do contain some useful signal!

## exercise  : TGCA BRCA prognostic data


1. run the following cell to load the TGCA BRCA prognostic data

In [ ]:
from sklearn.cluster import AgglomerativeClustering
import pandas as pd

def drop_correlated_features( X , threshold = 0.9 ):
    """
    Args:
        - X (pd.DataFrame) : n,p feature matrix
        - threshold (float) : absolute correlation threshold group variables
    
    Returns:
        - pd.DataFrame : X with only the selected variables
        - dict : keys are the selected features , values are the list of features in the corresponding feature cluster 
    """
    
    corr_threshold = 0.9

    metric = 1 - X.corr().abs()

    HC = AgglomerativeClustering( n_clusters=None , metric='precomputed', linkage = 'single' , distance_threshold = (1-corr_threshold) )
    HC.fit(metric)

    variable_clusters = pd.Series( HC.labels_  , index = X.columns)

    cluster_to_features = variable_clusters.index.groupby(variable_clusters)

    ## keys are the selected feature in the cluster, values are the list of features in the cluster
    selected_features_to_features = { v[0]:list(v) for v in cluster_to_features.values() }

    return X.loc[:,selected_features_to_features.keys()] ,  selected_features_to_features 




In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectPercentile
import numpy as np

## loading data
df_xpr = pd.read_csv("../data/TGCA_BRCA_expression_matrix.TPM.csv.gz" , index_col = 0)

df_clinical = pd.read_csv("../data/TGCA_BRCA_clinical_filtered.small.csv",index_col=0)
df_clinical = pd.get_dummies( df_clinical , drop_first=True)

## y is the poor_diagnosis
y = df_clinical.poor_prognosis

## ensuring the expression data is properly ordered
X_xpr = df_xpr.loc[ :, df_clinical.index].transpose() 

## selecting top 1% most variable genes
VT = SelectPercentile( score_func = lambda x,_ : np.var(x , axis = 0) ,
                       percentile = 1
                     )

X = pd.DataFrame( VT.fit_transform(X_xpr), columns=VT.get_feature_names_out() , index = X_xpr.index )
X , features_to_features_cluster = drop_correlated_features( X , threshold = 0.9 )


# adding age and sex to the set of features
X = pd.concat( [ df_clinical[['demographic.days_to_birth','demographic.sex_at_birth_male']] , X ] , axis=1 )

X.head()

 2. use cross-validation to estimate the accuracy of a random forest on this dataset
 3. use boruta to select all-relevant variables
 4. keep only the features selected by boruta and use cross-validation to estimate the accuracy of a random forest 
 5. run Boruta again, but with the `perc` parameter to 80 instead of 100. How many more variables does it select?

 ... correction ...

## training a Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline


rf = RandomForestClassifier(n_jobs=-1)

rf_grid = GridSearchCV(rf,
                       {"ccp_alpha" : np.logspace(-6,-3,20)},
                       scoring='accuracy'
                      )
%time rf_grid.fit(X,y)

In [ ]:
rf_grid.best_params_

In [ ]:
rf_grid.best_score_

In [ ]:
RF_fi = rf_grid.best_estimator_.feature_importances_
print( f"{( RF_fi != 0  ).sum()}/{X.shape[1]} features with non-zero importances")

## training a Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


ppl = Pipeline([('scale' , StandardScaler()),
                ('model' , LogisticRegression(l1_ratio=1.0,solver = 'liblinear'))
               ])

lr_grid = GridSearchCV(ppl,
                       {"model__C" : np.logspace(-5,5,100)}
                      )
%time lr_grid.fit(X,y)

In [ ]:
lr_grid.best_params_

In [ ]:
lr_grid.best_score_

In [ ]:
num_non_null = ( lr_grid.best_estimator_['model'].coef_ != 0 ).sum()

print( f"{num_non_null}/{X.shape[1]} features with non-zero importances")

## Borutapy

In [ ]:
from boruta import BorutaPy

rf = RandomForestClassifier(n_jobs=-1, ccp_alpha = rf_grid.best_params_['ccp_alpha'])
feat_selector = BorutaPy(rf, n_estimators='auto', verbose=2, random_state=123, max_iter = 250 )

In [ ]:
%time feat_selector.fit(X, y)

In [ ]:
## keep only the confirmed
Xf = feat_selector.transform( np.array(X) )
%time cross_val_score(rf, Xf,y).mean()

In [ ]:
## keep only confirmed + tentative
Xf = feat_selector.transform( np.array(X) , weak = True )
%time cross_val_score(rf, Xf,y).mean()

In [ ]:
from boruta import BorutaPy

rf = RandomForestClassifier(n_jobs=-1, ccp_alpha = rf_grid.best_params_['ccp_alpha'])
feat_selector = BorutaPy(rf, n_estimators='auto', verbose=2, random_state=123, max_iter = 250 , perc=80)

%time feat_selector.fit(X, y)

In [ ]:
## keep only the confirmed
Xf = feat_selector.transform( np.array(X) )
%time cross_val_score(rf, Xf,y).mean()

In [ ]:
X.columns[ feat_selector.support_ ]